# Chapter 9: Performance Tips & Best Practices

Writing efficient Pandas code is crucial for large datasets.

1. Vectorization vs `apply` vs `iterrows` (benchmarks)
2. Memory optimization
3. Method chaining with `.pipe()`
4. `eval()` and `query()` for large DataFrames
5. When to use NumPy directly
6. Common anti-patterns and fixes

In [ ]:
import pandas as pd
import numpy as np
import time

np.random.seed(42)

## 9.1 Vectorization vs `apply` vs `iterrows`

**Golden rule**: Always prefer vectorized operations. They are orders  
of magnitude faster because they run in compiled C/NumPy code.

In [ ]:
# Create a large DataFrame for benchmarking
n = 100_000
df = pd.DataFrame({
    'a': np.random.randn(n),
    'b': np.random.randn(n),
})

# Task: compute a + b * 2 for each row

In [ ]:
# Method 1: Vectorized (BEST)
start = time.perf_counter()
result1 = df['a'] + df['b'] * 2
t_vectorized = time.perf_counter() - start
print(f'Vectorized:  {t_vectorized*1000:.2f} ms')

In [ ]:
# Method 2: apply (SLOW)
start = time.perf_counter()
result2 = df.apply(lambda row: row['a'] + row['b'] * 2, axis=1)
t_apply = time.perf_counter() - start
print(f'apply:       {t_apply*1000:.2f} ms ({t_apply/t_vectorized:.0f}x slower)')

In [ ]:
# Method 3: iterrows (WORST)
start = time.perf_counter()
result3 = []
for idx, row in df.head(10000).iterrows():  # only 10k rows — too slow for 100k!
    result3.append(row['a'] + row['b'] * 2)
t_iterrows = time.perf_counter() - start
# Extrapolate to full dataset
t_iterrows_est = t_iterrows * (n / 10000)
print(f'iterrows:    {t_iterrows_est*1000:.2f} ms (estimated, {t_iterrows_est/t_vectorized:.0f}x slower)')

In [ ]:
# Summary
print('\n🏆 Performance Ranking:')
print(f'  1. Vectorized: {t_vectorized*1000:.2f} ms     ← ALWAYS use this')
print(f'  2. apply:      {t_apply*1000:.2f} ms   ← Use only when necessary')
print(f'  3. iterrows:   {t_iterrows_est*1000:.2f} ms  ← NEVER use this')

## 9.2 Memory Optimization

In [ ]:
# Check memory usage
df_mem = pd.DataFrame({
    'int_col': np.random.randint(0, 100, 100_000),
    'float_col': np.random.randn(100_000),
    'category_col': np.random.choice(['A', 'B', 'C', 'D', 'E'], 100_000),
    'bool_col': np.random.choice([True, False], 100_000),
})

print('Before optimization:')
print(df_mem.memory_usage(deep=True))
print(f'\nTotal: {df_mem.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Optimization 1: Category dtype for low-cardinality strings
df_mem['category_col'] = df_mem['category_col'].astype('category')

# Optimization 2: Downcast integers
df_mem['int_col'] = pd.to_numeric(df_mem['int_col'], downcast='integer')

# Optimization 3: Downcast floats
df_mem['float_col'] = pd.to_numeric(df_mem['float_col'], downcast='float')

print('After optimization:')
print(df_mem.memory_usage(deep=True))
print(f'\nTotal: {df_mem.memory_usage(deep=True).sum() / 1024:.1f} KB')

In [ ]:
# Category dtype — huge savings for repeated strings
s_str = pd.Series(['apple', 'banana', 'cherry'] * 10000)
s_cat = s_str.astype('category')

print(f'String memory: {s_str.memory_usage(deep=True) / 1024:.1f} KB')
print(f'Category memory: {s_cat.memory_usage(deep=True) / 1024:.1f} KB')
print(f'Savings: {(1 - s_cat.memory_usage(deep=True) / s_str.memory_usage(deep=True)) * 100:.1f}%')

## 9.3 Method Chaining with `.pipe()`

Method chaining produces readable, functional-style transformations.

In [ ]:
# Without chaining (procedural style)
df = pd.DataFrame({
    'name': ['alice', 'bob', 'charlie', 'diana'],
    'score': [85, 90, 78, 92],
    'dept': ['eng', 'mkt', 'eng', 'hr']
})

# Procedural:
df_temp = df.copy()
df_temp['name'] = df_temp['name'].str.title()
df_temp['dept'] = df_temp['dept'].str.upper()
df_temp = df_temp[df_temp['score'] >= 80]
df_temp = df_temp.sort_values('score', ascending=False)
df_temp

In [ ]:
# With method chaining — cleaner and more readable!
result = (
    df
    .assign(
        name=lambda x: x['name'].str.title(),
        dept=lambda x: x['dept'].str.upper(),
    )
    .query('score >= 80')
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)
result

In [ ]:
# .pipe() — apply a custom function in a chain
def add_grade(df):
    """Add a letter grade column."""
    df = df.copy()
    df['grade'] = pd.cut(
        df['score'],
        bins=[0, 70, 80, 90, 100],
        labels=['D', 'C', 'B', 'A']
    )
    return df

def add_rank(df, col='score'):
    """Add a rank column."""
    df = df.copy()
    df['rank'] = df[col].rank(ascending=False).astype(int)
    return df

# Chain custom functions with .pipe()
final = (
    df
    .assign(name=lambda x: x['name'].str.title())
    .pipe(add_grade)
    .pipe(add_rank)
    .sort_values('rank')
)
final

## 9.4 `eval()` and `query()` for Large DataFrames

For very large DataFrames, `eval()` and `query()` can be faster  
because they avoid creating intermediate arrays.

In [ ]:
n = 1_000_000
big_df = pd.DataFrame({
    'a': np.random.randn(n),
    'b': np.random.randn(n),
    'c': np.random.randn(n),
})

# Standard way
start = time.perf_counter()
result1 = big_df['a'] + big_df['b'] * big_df['c']
t1 = time.perf_counter() - start

# eval way
start = time.perf_counter()
result2 = big_df.eval('a + b * c')
t2 = time.perf_counter() - start

print(f'Standard: {t1*1000:.2f} ms')
print(f'eval:     {t2*1000:.2f} ms')
print(f'Results match: {np.allclose(result1, result2)}')

In [ ]:
# eval for creating new columns
big_df.eval('d = a + b * c', inplace=True)
big_df.head()

## 9.5 When to Use NumPy Directly

In [ ]:
# For pure numerical computation, NumPy can be faster
# because it avoids Pandas' index-alignment overhead

s = pd.Series(np.random.randn(1_000_000))

# Pandas
start = time.perf_counter()
_ = s.sum()
t_pd = time.perf_counter() - start

# NumPy
start = time.perf_counter()
_ = np.sum(s.values)
t_np = time.perf_counter() - start

print(f'Pandas sum: {t_pd*1000:.3f} ms')
print(f'NumPy  sum: {t_np*1000:.3f} ms')
print('\nTip: Use .values or .to_numpy() to get the underlying array')

## 9.6 Common Anti-Patterns & Fixes

### ❌ Anti-Pattern 1: Growing a DataFrame in a loop

In [ ]:
# ❌ BAD — appending rows in a loop (creates a new DataFrame each time!)
# result = pd.DataFrame()
# for i in range(1000):
#     row = pd.DataFrame({'a': [i], 'b': [i*2]})
#     result = pd.concat([result, row])  # Very slow!

# ✅ GOOD — collect rows in a list, create DataFrame once
rows = []
for i in range(1000):
    rows.append({'a': i, 'b': i * 2})
result = pd.DataFrame(rows)
print(f'Shape: {result.shape}')

### ❌ Anti-Pattern 2: Using iterrows for computation

In [ ]:
# ❌ BAD
# for idx, row in df.iterrows():
#     df.loc[idx, 'c'] = row['a'] + row['b']

# ✅ GOOD — vectorized
result['c'] = result['a'] + result['b']
print('Use vectorized operations!')

### ❌ Anti-Pattern 3: Not using appropriate dtypes

In [ ]:
# ❌ BAD — storing categories as strings
# df['status'] = ['active', 'inactive', 'active', ...]  # object dtype

# ✅ GOOD — use category dtype
df_demo = pd.DataFrame({
    'status': pd.Categorical(['active', 'inactive', 'active', 'pending'] * 1000)
})
print(f'Memory with category: {df_demo.memory_usage(deep=True).sum()} bytes')

df_demo2 = pd.DataFrame({
    'status': ['active', 'inactive', 'active', 'pending'] * 1000
})
print(f'Memory with strings:  {df_demo2.memory_usage(deep=True).sum()} bytes')

### ❌ Anti-Pattern 4: Chained indexing for assignment

In [ ]:
# ❌ BAD — chained indexing
# df[df['a'] > 0]['b'] = 999  # May not modify df!

# ✅ GOOD — use .loc
result.loc[result['a'] > 500, 'b'] = 999
print('Always use .loc for conditional assignment!')

## 📝 Performance Summary

| Tip | Impact |
|-----|--------|
| **Vectorize** | 10-100x faster than `apply` |
| **Category dtype** | Up to 90% memory savings |
| **Downcast numerics** | 50-75% memory savings |
| **Method chaining** | Cleaner, more maintainable code |
| **Collect in list** | Much faster than concat-in-loop |
| **eval/query** | Faster for complex expressions on large DataFrames |
| **Use .loc** | Avoid SettingWithCopyWarning |